## ML_1047: DPO (Direct Preference Optimization) Loss

**Difficulty**: Hard | **TorchCode**: #37

### Problem
Implement the DPO loss for aligning LLMs from human preferences without a separate reward model. Maximize the margin between chosen and rejected log-probability ratios under the reference policy.

### Formula
$$\mathcal{L}_{\text{DPO}} = -\mathbb{E}\!\left[\log\sigma\!\left(\beta\left[(\log\pi_\theta(y_w) - \log\pi_{\text{ref}}(y_w)) - (\log\pi_\theta(y_l) - \log\pi_{\text{ref}}(y_l))\right]\right)\right]$$

In [ ]:
import torch
import torch.nn.functional as F

def dpo_loss(policy_chosen_logps, policy_rejected_logps,
             ref_chosen_logps, ref_rejected_logps, beta=0.1):
    chosen_rewards  = beta * (policy_chosen_logps  - ref_chosen_logps)
    rejected_rewards = beta * (policy_rejected_logps - ref_rejected_logps)
    return -F.logsigmoid(chosen_rewards - rejected_rewards).mean()

In [ ]:
import torch
import torch.nn.functional as F

# --- Test 1: Easy pair — small loss ---
chosen = torch.tensor([0.0, 0.0]); rejected = torch.tensor([-10.0, -10.0])
ref_c = torch.tensor([-1.0, -1.0]); ref_r = torch.tensor([-1.0, -1.0])
loss = dpo_loss(chosen, rejected, ref_c, ref_r, beta=0.1)
assert loss.dim() == 0 and loss.item() < 0.5

# --- Test 2: Hard pair — large loss ---
loss2 = dpo_loss(torch.tensor([-10.0]), torch.tensor([0.0]),
                 torch.tensor([-1.0]), torch.tensor([-1.0]), beta=0.1)
assert loss2.item() > 0.5

# --- Test 3: Gradient flow ---
c = torch.randn(4, requires_grad=True); r = torch.randn(4, requires_grad=True)
dpo_loss(c, r, torch.randn(4), torch.randn(4)).backward()
assert c.grad is not None and r.grad is not None

# --- Test 4: Mathematical correctness ---
torch.manual_seed(0)
c, r, rc, rr = torch.randn(3), torch.randn(3), torch.randn(3), torch.randn(3)
beta = 0.5
loss = dpo_loss(c, r, rc, rr, beta=beta)
ref = -torch.log(torch.sigmoid(beta * ((c - rc) - (r - rr)))).mean()
assert torch.allclose(loss, ref, atol=1e-5)

# --- Test 5: Beta scaling affects loss ---
torch.manual_seed(0)
c, r, rc, rr = torch.randn(4), torch.randn(4), torch.randn(4), torch.randn(4)
assert not torch.allclose(dpo_loss(c, r, rc, rr, beta=0.1), dpo_loss(c, r, rc, rr, beta=1.0))

print('All tests passed!')